# Forest Cover Type Classification

One of the classic big tabular datasets: 581,012 patches of Colorado forest, each described by elevation, slope, soil type and wilderness area. Predict which of **7 tree cover types** grows there. We train on a stratified 120,000-row sample so the notebook stays quick.


In [1]:
import io, urllib.request, zipfile, gzip
import numpy as np
import pandas as pd

def download(url):
    req = urllib.request.Request(url, headers={'User-Agent': 'Mozilla/5.0'})
    with urllib.request.urlopen(req, timeout=180) as r:
        return r.read()
from sklearn.ensemble import RandomForestClassifier
from sklearn.linear_model import LogisticRegression
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.pipeline import make_pipeline
from sklearn.metrics import accuracy_score, classification_report

raw = download('https://archive.ics.uci.edu/static/public/31/covertype.zip')
zf = zipfile.ZipFile(io.BytesIO(raw))
name = [n for n in zf.namelist() if 'covtype.data' in n][0]
df = pd.read_csv(io.BytesIO(gzip.decompress(zf.read(name))), header=None)
print(df.shape)
df.head(3)

(581012, 55)


,0,1,2,3,4,5,6,7,8,9,...,45,46,47,48,49,50,51,52,53,54
0,2596,51,3,258,0,510,221,232,148,6279,...,0,0,0,0,0,0,0,0,0,5
1,2590,56,2,212,-6,390,220,235,151,6225,...,0,0,0,0,0,0,0,0,0,5
2,2804,139,9,268,65,3180,234,238,135,6121,...,0,0,0,0,0,0,0,0,0,2


In [2]:
y_all = df.iloc[:, -1]; X_all = df.iloc[:, :-1]
Xs, _, ys, _ = train_test_split(X_all, y_all, train_size=120000, random_state=42, stratify=y_all)
Xtr, Xte, ytr, yte = train_test_split(Xs, ys, test_size=0.25, random_state=42, stratify=ys)
rf = RandomForestClassifier(n_estimators=200, random_state=42, n_jobs=-1).fit(Xtr, ytr)
lr = make_pipeline(StandardScaler(), LogisticRegression(max_iter=1000)).fit(Xtr, ytr)
print(f'Random Forest accuracy: {accuracy_score(yte, rf.predict(Xte)):.4f}')
print(f'Logistic Regression accuracy: {accuracy_score(yte, lr.predict(Xte)):.4f}')
print(classification_report(yte, rf.predict(Xte)))

Random Forest accuracy: 0.9082
Logistic Regression accuracy: 0.7240


              precision    recall  f1-score   support

           1       0.92      0.89      0.91     10938
           2       0.90      0.95      0.92     14628
           3       0.88      0.92      0.90      1846
           4       0.87      0.83      0.85       142
           5       0.91      0.50      0.65       490
           6       0.88      0.76      0.81       897
           7       0.95      0.86      0.90      1059

    accuracy                           0.91     30000
   macro avg       0.90      0.82      0.85     30000
weighted avg       0.91      0.91      0.91     30000



In [3]:
imp = pd.Series(rf.feature_importances_).sort_values(ascending=False)
print('Top features (first 10 of 54 are the continuous ones: Elevation, Aspect, Slope, distances, hillshades):')
print(imp.head(10))

Top features (first 10 of 54 are the continuous ones: Elevation, Aspect, Slope, distances, hillshades):
0    0.237485
5    0.105669
9    0.097099
3    0.060504
4    0.058686
1    0.053900
7    0.049978
8    0.048251
6    0.047950
2    0.039077
dtype: float64
